# Tutorial: WorldQuant Brain — Bigdata.com Search API

Search news by meaning—not just keywords—and filter by entity, time, and sentiment to build reproducible, point-in-time queries for research and backtesting. This tutorial shows how to use the Search API for semantic document search with entity and keyword filters.

**What you'll learn**
- Review the index and understand the underlying data structure
- Run a semantic search and filter by entity, time, and sentiment
- Study and tune all key search parameters, including the effects of `freshness` on results
- Interpret responses (documents, chunks, relevance, sentiment)
- Optimize `max_chunks` and `source_boost` for the best balance of quality and coverage

**Why Bigdata.com Search**
- Semantic search over structured news with entity and sentiment metadata
- Built for research and backtesting (e.g. point-in-time, configurable time windows)
- Relevance-ranked chunks and source ranks to control quality and coverage

**Example use cases:** alpha/signal research, event studies, sentiment or thematic filters, competitive monitoring.

**Signals:** Search results are the raw chunks you will later label and aggregate into signals; see Workflow_example for the full pipeline.

**In this notebook**
1. **Search API** — run a basic semantic search (entity + time), interpret the response, and understand the response structure.
2. **Filters and ranking** — filters (entity, keyword, time, sentiment, source) and ranking parameters (reranker, threshold, source_boost, freshness_boost).
3. **Query chunk size** — tune `max_chunks` for quality vs coverage using the Volume API and grid search.
4. **Source boosting** — use `source_boost` to favor premium sources and the trade-off with coverage.
5. **Freshness** — effect of `freshness_boost` on the time distribution of results.

**Prerequisites:** `.env` file with `BRAIN_EMAIL` and `BRAIN_PASSWORD` (see `.env.example`). For access, see the hackathon or [WorldQuant Brain](https://www.worldquantbrain.com/) documentation.

**Time:** about 20–30 minutes.

**Endpoint documentation:** [Search API reference](https://docs.bigdata.com/api-reference/search/search-documents)

In [ ]:
import requests
import json
from session import BrainSession
from print_helpers import print_search_results

## Setup & Configuration

In [ ]:
# Load credentials from .env file
import os
from dotenv import load_dotenv
load_dotenv()

# WorldQuant Brain API Configuration
API_BASE_URL = "https://api.worldquantbrain.com"
EMAIL = os.getenv("BRAIN_EMAIL")
PASSWORD = os.getenv("BRAIN_PASSWORD")

# API Endpoints
SEARCH_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/search"
VOLUME_ENDPOINT = f"{SEARCH_ENDPOINT}/volume"  # Used for optimization comparison

session = BrainSession(
    base_url=API_BASE_URL,
    email=EMAIL,
    password=PASSWORD
)

## Key considerations before starting

The provider APIs do not have full coverage for parameter validation—double-check your request against the [API reference](https://docs.bigdata.com/api-reference) to avoid issues from typos or invalid values. 


## 1. Search API

The Search API lets you search for documents based on text queries. It uses **semantic** search—matching by meaning, not just keywords—so a query like "cloud computing market growth" can match related phrasing. You can filter by timestamp, entities, sources, and more.

We start with a basic example: chunks in 2021 that semantically match `cloud computing market growth` and where **Apple** OR **Microsoft** are mentioned. The endpoint uses **ENTITY_ID** for companies (e.g. from a PiT Russell 3000 mapping); other notebooks show how to map company names to IDs. 

In [ ]:
# Date range for examples
START_DATE = "2021-01-01"
END_DATE = "2021-12-31"
TEXT = "the company expect revenue driven by cloud computing market growth"
ENTITY_ID = ["D8442A","228D42"]  # Apple Inc, Microsoft

Before we run the query, two parameters are worth setting explicitly.

`auto_enrich_filters`
- When set to `True`, entities are automatically extracted from the text using NLP and added to the `any_of` filter by default. This behavior can introduce unintended results, especially when using common entities, like places.
- When set to `False` (**recommended**), you retain full control over entity-based filtering.

`freshness_boost`

Set this parameter to `0` for most applications. Increasing it gives higher ranking to more recent documents—which is helpful for generating forward-looking, real-time signals. For retrieval or backtesting use cases where you want an unbiased set of top results (not favoring newer content), use the default of `0`.



In [ ]:
# Semantic search with entity filter (Apple OR Microsoft)
search_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z"
            },
            "entity": {
                "any_of": ENTITY_ID,
            },
        },
        "ranking_params": {
            "freshness_boost": 0
        },
        "max_chunks": 10
    }
}

response = session.post(SEARCH_ENDPOINT, json=search_query)
data = response.json()
print_search_results(response, data)

You'll see documents with chunks, relevance, and sentiment—filtered by your entity and time range. Try changing the query text or entities and re-run the search above to see how results change.

### 1.1 Response Structure

Understanding what the API returns.

In short: you get a list of documents, each with chunks (text, relevance, sentiment, detections).

**Response contains three main sections:**

**1. `results`:** List of documents matching your query
- `id`: Unique document identifier
- `headline`: Document title
- `timestamp`: Publication date/time (ISO 8601 format)
- `source`: {id, name, rank} - Source information
- `url`: Link to original article (when available)
- `chunks`: List of relevant text segments

**2. `metadata`:** Request information
- `request_id`: Unique identifier for this request
- `timestamp`: When the request was processed

**3. `usage`:** API consumption metrics
- `api_query_units`: Credits consumed by this query

**CHUNK STRUCTURE:**

Each chunk contains:
- `cnum`: Chunk number within the document
- `text`: The actual text content
- `relevance`: Score (0-1) indicating match quality
- `sentiment`: Overall sentiment score (-1 to 1, where 1=positive, -1=negative)
  - **NOTE:** This is the overall sentiment of the chunk, NOT entity-specific
- `detections`: List of entities/topics mentioned
  - `id`: Entity/topic identifier (6-char hex for entities, path for topics)
  - `type`: "entity" or "topic"
  - `start/end`: Character positions in text

In [ ]:
# Show response structure
print(json.dumps(data, indent=2))

## 2. Advanced Options in the Search API: Filters and Ranking

Next, we look at **filters and ranking** in detail—which documents are returned and how they are ordered. The Search API gives you fine-grained control over *which* documents are returned (**filters**) and *how* they are ordered and scored (**ranking parameters**). This section explains each option in detail so you can build precise, reproducible queries—especially important for backtesting and point-in-time research.

You can restrict results in several ways:

- **By entity:** Include or exclude specific organizations, people, or other entities using the `entity` filter with `any_of` (at least one must appear), `all_of` (all must appear), or `none_of` (none may appear). Entity IDs are used (e.g. from a PiT Russell 3000 mapping or entity lookup). Optional `search_in`: **HEADLINE**, **BODY**, or **ALL** (defaults to **ALL**).
- **By keyword:** Target documents that contain certain terms using the `keyword` filter with the same `any_of` / `all_of` / `none_of` logic. Optional `search_in`.
- **By time:** Restrict to a time window with the `timestamp` filter using `start` and `end` in ISO 8601 format.
- **By sentiment:** Keep only chunks whose overall sentiment falls in a given range (e.g. only negative or only positive).
- **By source:** Limit results to specific publishers using the `source` filter (source ID).
- **By document type:** For the hackathon, only `news` is available; use `document_type` to ensure you only get news documents.

All of these filters can be combined; the API applies them together (e.g. timestamp AND entity AND keyword). Below is a concise reference with didactic notes.

The same filters are available on the **Volume** and **Co-mentions** endpoints (each endpoint returns different payloads—documents, volume counts, or co-mentioned entities—but filter semantics are shared).

---

### Recommended filters (what gets returned)

| Filter | Description |
|--------|-------------|
| `timestamp` | Time range: `start` and `end` in ISO 8601 format (e.g. `YYYY-MM-DDTHH:MM:SSZ`). Only documents whose publication time falls in this range are returned. **Required for most queries.** |
| `entity` | Filter by entity IDs. Use `any_of`, `all_of`, or `none_of` with a list of IDs. Optional `search_in`: **HEADLINE** (only in headlines), **BODY** (only in body text), or **ALL** (both). Defaults to **ALL** if not provided. Example: `"entity": { "search_in": "HEADLINE", "any_of": ["D8442A"] }`. |
| `keyword` | Literal keyword matching with `any_of`, `all_of`, or `none_of`. Same optional `search_in` as `entity`: **HEADLINE**, **BODY**, or **ALL**. Defaults to **ALL** if not provided. |
| `source` | Restrict to a specific source (publisher) by source ID. |
| `sentiment` | Filter by the chunk's overall sentiment score from **-1.0** (negative) to **1.0** (positive). This is chunk-level sentiment, not entity-specific. You can pass **multiple ranges** (e.g. `[{"min": -1.0, "max": -0.3}, {"min": 0.3, "max": 1.0}]`) to get either negative or positive chunks. |
| `document_type` | For the hackathon, only **news** is available; use this to restrict to news documents. |

**Other filters:** The **topic** filter allows you to restrict results to only those chunks in which a specific event from the RavenPack Analytics taxonomy was detected. It supports the same optional `search_in`: **HEADLINE**, **BODY**, or **ALL** (defaults to **ALL**). This relies on RavenPack's event detection methodology, which is useful if your analysis needs to focus on well-defined event types, especially in finance. However, its use-case is limited by the scope of the RavenPack taxonomy—it is relatively broad but finance-specific, and does not offer the full flexibility or coverage of freeform semantic matching. For most general queries, the topic filter is not recommended.

---

### Ranking parameters (order and relevance trimming)

These parameters affect how results are ordered and which chunks are kept after retrieval:

| Parameter | Description |
|-----------|-------------|
| `reranker` | When enabled, a cross-encoder re-ranker recomputes semantic relevance for each chunk and reorders the results. This mechanism is especially valuable for LLM applications where you want to retrieve only the top most-relevant chunks or those above a specific similarity threshold. Set `enabled: false` to disable the re-ranker. When enabled, the `threshold` parameter below takes effect. |
| `threshold` | Relevance score cutoff (**0.0–1.0**, default **0.2**). When the cross-encoder re-ranker is enabled, only chunks scoring above this threshold are retained. **Higher threshold values yield fewer, but more relevant, results.** Useful for filtering to the highest-similarity chunks in retrieval-augmented generation (RAG) and LLM workflows. |
| `source_boost` | Multiplier for premium sources (**0–10**, default **1.0**). Use to upweight premium publishers. |
| `freshness_boost` | **Must be set to 0** for the hackathon (and for point-in-time / historical research). Otherwise it boosts more recent documents; for 2021–2022 backtesting, set to **0** so time is not weighted. |

---

### Important notes

- **timestamp:** Use ISO 8601 format (e.g. `YYYY-MM-DDTHH:MM:SSZ`). Required for most queries.
- **entity / keyword / topic:** All three support the optional `search_in` parameter: **HEADLINE** (only in headlines), **BODY** (only in body text), or **ALL** (both). Defaults to **ALL** if not provided.
- **sentiment:** Applies to the **overall chunk** sentiment, not to a specific entity. Sentiment is provided at the chunk level so you can filter by tone without extra NLP. Multiple ranges are OR'd (e.g. negative OR positive).
- **topic:** Not recommended.
- **freshness_boost:** Recommended value is **0** to ignore publishing time (essential for point-in-time research).
- **reranker:** Use `enabled: false` to turn off re-ranking. When enabled, `threshold` (0.0–1.0, default 0.2) filters by relevance; higher = fewer, more relevant results.
- **Parameter validation:** The provider APIs do not have full coverage for parameter validation; typos or invalid values can lead to unexpected behavior. For full schemas and options, see the [official API documentation](https://docs.bigdata.com).

## 3. Query Chunk Size: Best Practices

We now turn to **query chunk size**—how many chunks to request and how that affects quality and coverage. This section demonstrates how to optimize search queries by:
1. Using Volume API to understand data distribution
2. Understanding the relationship between `max_chunks` parameter and actual chunks retrieved
3. Avoiding inefficient queries with excessive `max_chunks` values

**Key Insights:**

**1. CHUNKS ARE RANKED BY RELEVANCE**
- The API returns chunks ordered from MOST relevant to LEAST relevant
- Increasing `max_chunks` adds progressively LESS relevant chunks
- This increases NOISE in your results, especially within the same day/period

**2. DIMINISHING RETURNS**
- Increasing `max_chunks` does NOT linearly increase chunks retrieved
- The efficiency curve shows saturation: beyond a threshold, you're just requesting more chunks that don't exist or aren't relevant enough

**3. QUALITY vs QUANTITY TRADE-OFF**
- Higher `max_chunks` = more data but lower average quality
- Lower `max_chunks` = less data but higher average quality
- Find the optimal balance based on your use case

**For signals:** Skewed or low coverage can bias downstream signals; these parameters help you balance coverage and quality for signal construction.

In [ ]:
from api_helpers import (
    get_volume_dataframe,
    grid_parameter_search,
    plot_chunks_vs_max_chunks
)

# Get Volume Data 
volume_df = get_volume_dataframe(
    session=session,
    volume_endpoint=VOLUME_ENDPOINT,
    text=TEXT,
    entities=ENTITY_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    entity_mode="any_of",
    freshness_boost=0
)

In [ ]:
# Grid Search - Test different max_chunks values
MAX_CHUNKS_VALUES = [25, 75, 200, 400, 600, 800, 1000]

grid_df, grid_results = grid_parameter_search(
    session=session,
    search_endpoint=SEARCH_ENDPOINT,
    text=TEXT,
    entities=ENTITY_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    max_chunks_values=MAX_CHUNKS_VALUES,
    freshness_values=[0],
    entity_mode="any_of"
)

# Calculate efficiency
grid_df['efficiency_%'] = (grid_df['chunks'] / grid_df['max_chunks'] * 100).round(1)
grid_df

 Although the requested period contains more than 1000 potential chunks, the retrieval process applies internal deduplication, which significantly reduces the number of unique chunks actually extracted. This means that, despite increasing the `max_chunks` parameter, many near-duplicate or redundant pieces of information are filtered out, reflecting the true diversity of content available. 
 
 In the following plots, the number of unique chunks retrieved for each `max_chunks` value is visualized alongside the actual weekly volume of available content. This allows for a direct comparison between the theoretical maximum volume and the deduplicated, real-world data returned by the API, highlighting both the behavior of the deduplication mechanism and the relationship between retrieval settings and actual content volume.

In more advanced notebooks, we will explore strategies to optimize search settings when constrained to a limited number of chunks. The goal will be to maintain a representative distribution of content across different entities and time periods.

In [ ]:
# Temporal distribution plot - shows how documents are distributed over time
from api_helpers import plot_freshness_comparison

volume_results_list = volume_df.to_dict('records') if volume_df is not None and not volume_df.empty else None

fig_temporal = plot_freshness_comparison(
    results=grid_results,
    freshness_values=[0],
    max_chunks_values=MAX_CHUNKS_VALUES,
    text=TEXT,
    start_date=START_DATE,
    end_date=END_DATE,
    volume_results=volume_results_list,
    entity_id=ENTITY_ID,
    group_by_week=True
)
fig_temporal.show()

In [ ]:
# Efficiency plot - shows chunks retrieved vs max_chunks
fig = plot_chunks_vs_max_chunks(
    results=grid_results,
    max_chunks_values=MAX_CHUNKS_VALUES,
    freshness_boost=0,
    text=TEXT,
    start_date=START_DATE,
    end_date=END_DATE
)

# Display plot
try:
    import matplotlib.pyplot as plt
    plt.show()
except:
    pass

## 4. Source Boosting: Controlling Source Quality

Next, we look at **source boosting**—how to favor premium sources and the trade-off with coverage. The `source_boost` parameter allows you to prioritize higher-ranked sources. Source ranks let you favor premium or trusted publishers when it matters.

**Key Insights:**

**THE QUALITY vs COVERAGE TRADE-OFF**
- Higher `source_boost` = higher source quality BUT fewer total results
- Too high `source_boost` can drastically REDUCE available results
- You may miss relevant content from lower-ranked but still valid sources

In the following experiments, you will need to observe how varying the `source_boost` parameter affects both the quality and quantity of your search results. 
Pay close attention to the trade-off: as you increase source_boost, you should see results increasingly dominated by higher-ranked sources, but the overall coverage may decrease, potentially omitting some relevant lower-ranked sources.
Use the analysis and plots provided to understand which value best fits your goals—whether you want maximum reliability (higher boost) or broader coverage (lower boost).

**For signals:** The same trade-off applies when building signals: skewed coverage can bias your signal; tune `max_chunks` and `source_boost` to balance quality and coverage for your pipeline.

In [ ]:
from api_helpers import (
    plot_source_distribution,
    plot_source_rank_distribution,
    get_source_rank_summary
)

# Grid Search - Test different source_boost values (with freshness_boost=0)
SOURCE_BOOST_VALUES = [0, 1, 5, 10]

grid_source_df, results_source_raw = grid_parameter_search(
    session=session,
    search_endpoint=SEARCH_ENDPOINT,
    text=TEXT,
    entities=ENTITY_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    source_boost_values=SOURCE_BOOST_VALUES,
    max_chunks_values=[500],
    fixed_freshness_boost=0,
    entity_mode="any_of"
)

# Convert results format: (source_boost, max_chunks) -> source_boost
results_source = {sb: results_source_raw.get((sb, 500)) for sb in SOURCE_BOOST_VALUES}

In [ ]:
# Debug: Check source field structure
data_sample = results_source.get(0)
if data_sample and data_sample.get('results'):
    first_doc = data_sample['results'][0]
    print("Source field structure in first document:")
    print(json.dumps(first_doc.get('source', {}), indent=2))
    print(f"\nTotal docs for source_boost=0: {len(data_sample['results'])}")

In [ ]:
# Source Distribution Analysis
fig_bar, fig_line = plot_source_distribution(
    results_source=results_source,
    source_boost_values=SOURCE_BOOST_VALUES,
    top_n_sources=50,
    top_n_per_subplot=15
)
fig_bar.show()
fig_line.show()

In [ ]:
# Source Rank Distribution Analysis
fig_rank_bar, fig_rank_line = plot_source_rank_distribution(
    results_source=results_source,
    source_boost_values=SOURCE_BOOST_VALUES
)
fig_rank_bar.show()
fig_rank_line.show()

In [ ]:
# Summary DataFrame
rank_summary_df = get_source_rank_summary(results_source, SOURCE_BOOST_VALUES)
rank_summary_df

## 5. Effect of freshness on volume distribution

Finally, we compare **freshness_boost = 0** vs **freshness_boost = 10** and how that shifts the time distribution of results. This section compares **freshness_boost = 0** (no recency bias; recommended for historical/backtest) with **freshness_boost = 10** (strong boost for more recent documents). Higher freshness shifts which documents rank higher, so the *distribution* of retrieved documents over time changes even when the total available volume (from the Volume API) is the same. Below we run an experiment with `max_chunks=1000` for both freshness values and plot document counts over time alongside the Volume API baseline.

 The plots reveal a clear volume shift toward later dates when applying a high freshness_boost. This recency bias is useful for prioritizing newer information—such as for real-time or forward-looking signals—ensuring that more recent documents are surfaced more prominently than with no recency bias.

 For situations where you want to smooth time signals without re-retrieving chunks, you can instead apply exponential decay as a post-processing weighting technique. This efficiently shifts signal emphasis toward recent items by adjusting weights after retrieval, rather than running the search again with different freshness parameters.

In [ ]:
# Compare freshness_boost 0 vs 1: run grid over a subset of max_chunks
FRESHNESS_MAX_CHUNKS = [1000]

grid_df_freshness, grid_results_freshness = grid_parameter_search(
    session=session,
    search_endpoint=SEARCH_ENDPOINT,
    text=TEXT,
    entities=ENTITY_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    max_chunks_values=FRESHNESS_MAX_CHUNKS,
    freshness_values=[0, 10],
    entity_mode="any_of"
)

grid_df_freshness["efficiency_%"] = (grid_df_freshness["chunks"] / grid_df_freshness["max_chunks"] * 100).round(1)
grid_df_freshness

In [ ]:
# Plot document distribution over time: one subplot per freshness_boost (0 vs 1)
volume_results_list = volume_df.to_dict("records") if volume_df is not None and not volume_df.empty else None

fig_freshness = plot_freshness_comparison(
    results=grid_results_freshness,
    freshness_values=[0, 10],
    max_chunks_values=FRESHNESS_MAX_CHUNKS,
    text=TEXT,
    start_date=START_DATE,
    end_date=END_DATE,
    volume_results=volume_results_list,
    entity_id=str(ENTITY_ID),
    group_by_week=True
)
fig_freshness.show()



## 6.  Effect of reranker and threshold on relevance distribution

The **reranker** (cross-encoder) recomputes semantic relevance for each chunk and can filter chunks by a **threshold**. Comparing three configurations shows how the distribution of relevance scores changes:

1. **No reranker** (`enabled: false`) — chunks use the initial retrieval scores; you see the full spread.
2. **Reranker with threshold 0.2** — only chunks with relevance above 0.2 are kept; default for many RAG workflows.
3. **Reranker with threshold 0.7** — only high-relevance chunks are kept; fewer results, higher average quality.

Below we run the same query with each configuration and compare the **relevance distribution** (count, mean, and histogram) of returned chunks.

In [ ]:
# Base query (same for all three runs)
RERANKER_MAX_CHUNKS = 500  # request enough chunks to compare distributions

def run_search_reranker_config(session, search_endpoint, reranker_enabled, threshold=None, return_chunks=False):
    """Run search with given reranker config; 
    - If return_chunks: returns list of (relevance, full_text, chunk dict) tuples sorted by relevance descending
    - else: returns list of relevance scores"""
    ranking_params = {"freshness_boost": 0}
    if reranker_enabled:
        ranking_params["reranker"] = {"enabled": True, "threshold": threshold}
    else:
        ranking_params["reranker"] = {"enabled": False}

    search_query = {
        "query": {
            "text": TEXT,
            "auto_enrich_filters": False,
            "filters": {
                "timestamp": {
                    "start": f"{START_DATE}T00:00:00Z",
                    "end": f"{END_DATE}T23:59:59Z",
                },
                "entity": {"any_of": ENTITY_ID},
            },
            "ranking_params": ranking_params,
            "max_chunks": RERANKER_MAX_CHUNKS,
        }
    }
    response = session.post(search_endpoint, json=search_query)
    if response.status_code != 200:
        raise ValueError(f"Search failed: {response.status_code} - {response.text[:500]}")
    data = response.json()
    relevances = []
    chunks_info = []
    for doc in data.get("results", []):
        for chunk in doc.get("chunks", []):
            r = chunk.get("relevance")
            if r is not None:
                relevances.append(r)
                if return_chunks:
                    # Use full chunk text
                    chunk_text = chunk.get("text", "")
                    chunks_info.append((r, chunk_text, chunk))
    if return_chunks:
        # Sort by relevance (descending), then return tuples (relevance, full text)
        return sorted(chunks_info, key=lambda x: -x[0])
    return relevances

# Run four configurations and fetch chunks as well as just scores
relevances_no_reranker = run_search_reranker_config(session, SEARCH_ENDPOINT, reranker_enabled=False)
relevances_reranker_02 = run_search_reranker_config(session, SEARCH_ENDPOINT, reranker_enabled=True, threshold=0.2)
relevances_reranker_07 = run_search_reranker_config(session, SEARCH_ENDPOINT, reranker_enabled=True, threshold=0.7)
relevances_reranker_09 = run_search_reranker_config(session, SEARCH_ENDPOINT, reranker_enabled=True, threshold=0.9)

# For top 3 chunk tables, get full details with return_chunks=True
chunks_no_reranker = run_search_reranker_config(session, SEARCH_ENDPOINT, reranker_enabled=False, return_chunks=True)
chunks_reranker_02 = run_search_reranker_config(session, SEARCH_ENDPOINT, reranker_enabled=True, threshold=0.2, return_chunks=True)
chunks_reranker_07 = run_search_reranker_config(session, SEARCH_ENDPOINT, reranker_enabled=True, threshold=0.7, return_chunks=True)
chunks_reranker_09 = run_search_reranker_config(session, SEARCH_ENDPOINT, reranker_enabled=True, threshold=0.9, return_chunks=True)

# Summary stats
def summary(name, scores):
    if not scores:
        return f"{name}: 0 chunks"
    return f"{name}: n={len(scores)}, mean={sum(scores)/len(scores):.4f}, min={min(scores):.4f}, max={max(scores):.4f}"

print(summary("No reranker", relevances_no_reranker))
print(summary("Reranker threshold 0.2", relevances_reranker_02))
print(summary("Reranker threshold 0.7", relevances_reranker_07))
print(summary("Reranker threshold 0.9", relevances_reranker_09))

# Generate table with top 3 chunks for each method (full, not truncated)
import pandas as pd

def top_chunks_table(chunk_tuples, label):
    if not chunk_tuples:
        return pd.DataFrame(columns=["method", "rank", "relevance", "text"])
    top3 = chunk_tuples[:3]
    rows = []
    for i, (rel, txt, chunk) in enumerate(top3, 1):
        # Show the full chunk text (preserve newlines for wrapping)
        rows.append({
            "method": label,
            "rank": i,
            "relevance": f"{rel:.4f}",
            "text": txt
        })
    return pd.DataFrame(rows)

df_top = pd.concat([
    top_chunks_table(chunks_no_reranker, "No reranker"),
    top_chunks_table(chunks_reranker_02, "Reranker threshold 0.2"),
    top_chunks_table(chunks_reranker_07, "Reranker threshold 0.7"),
    top_chunks_table(chunks_reranker_09, "Reranker threshold 0.9"),
], axis=0).reset_index(drop=True)

print("\nTop 3 chunks for each method:")
display(df_top.style.set_properties(subset=["text"], **{'max-width': '520px', "white-space": "pre-wrap"}))

# Relevance distribution plot
import matplotlib.pyplot as plt
import numpy as np

# No KDE -- plot histograms, then add stepped line plots for each config
fig, ax = plt.subplots(figsize=(10, 7))
relevance_sets = [
    (relevances_no_reranker, "No reranker", "C0"),
    (relevances_reranker_02, "Reranker threshold 0.2", "C1"),
    (relevances_reranker_07, "Reranker threshold 0.7", "C2"),
    (relevances_reranker_09, "Reranker threshold 0.9", "C3"),
]

x_grid = np.linspace(0, 1, 500)
bin_edges = np.linspace(0, 1, 101)  # Finer y-axis resolution (bins of 0.01)

for values, label, color in relevance_sets:
    if values:
        # Plot histogram (transparent), remove label from histogram for no legend for areas
        ax.hist(values, bins=bin_edges, alpha=0.15, label=None, color=color, density=True)
        # Compute histogram (density)
        counts, edges = np.histogram(values, bins=bin_edges, density=True)
        centers = (edges[:-1] + edges[1:]) / 2
        # Plot as a step/line with more y-resolution
        ax.step(centers, counts, where="mid", label=f"{label} (line)", color=color, linewidth=2)
        # Optionally overlay the points for clarity
        ax.plot(centers, counts, 'o', color=color, markersize=2, alpha=0.6)

ax.set_xlabel("Relevance score")
ax.set_ylabel("Density")
ax.set_title(f"Relevance distribution by reranker config (query: \"{TEXT[:40]}...\" | max_chunks={RERANKER_MAX_CHUNKS})")
ax.set_ylim(bottom=0)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

  The plots above illustrate how the distribution of "relevance scores" changes under different reranker settings.
  As you lower the reranker threshold, lower-similarity chunks are filtered out, making the remaining set more focused and shifting scores toward higher relevance. 
Remember, because each retrieval independently reevaluates chunk relevance, scores can only be directly compared within the same reranker configuration.
 
  **Try it yourself:** Experiment with different reranker thresholds and observe how the relevance score distribution shifts. What threshold gives you the best balance between coverage and quality for your use case?

## Summary and next steps

**What we covered**
- Semantic search with entity, timestamp, and filter options
- Response structure (documents, chunks, relevance, sentiment)
- Tuning `max_chunks` and `source_boost` for quality vs coverage

**Next steps**
- **Volume API** ([Volume_API](../Volume_API/)) — understand data availability and distribution before searching
- **CoMentions API** ([CoMentions_API](../CoMentions_API/)) — entity co-occurrence and relationship discovery
- **Knowledge Graph API** ([Knowledge_Graph_API](../Knowledge_Graph_API/)) — explore entities and relationships
- **Full API reference:** [docs.bigdata.com](https://docs.bigdata.com)

You can experiment by changing `TEXT`, `ENTITY_ID`, or the date range and re-running the notebook to explore different queries.